# Lesson 1 - Exercise 1: Profile GPT-2 and Analyze Top Operators

**Goal:** Apply the profiling setup learned in the demo, run it on a smaller model (GPT-2), and practice accessing and interpreting the detailed profiler results to identify key performance bottlenecks.

**Task Overview:**
1.  Set the model name to `"gpt2"`.
2.  Re-run the CPU and GPU profiling sections, ensuring you capture results in `prof_cpu` and `prof_gpu`.
3.  Add code to print the detailed operator tables from the profiler results.
4.  Identify and list the Top 5 operators for both CPU and GPU.
5.  Interpret what these top operators likely represent.
6.  Compare the overall wall clock time of `gpt2` (this exercise) with `gpt2-medium` (from the demo or your own run if you did it).


## Imports

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.profiler
import time

## Load Model and Define Prompt

**TODO:**
- Set `model_name` variable to `"gpt2"`.
- The rest of the cell (loading tokenizer, model, setting pad token, and preparing inputs) can remain largely the same as the demo code.

In [3]:
# TODO: Define the model name for GPT-2 (the smallest variant)
model_name = "gpt2"  # Replace with "gpt2"

print(f"Loading model and tokenizer for: {model_name}...")
try:
    # TODO: Load the tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
except Exception as e:
    print(f"Error loading model {model_name}. Please ensure it's correct. Error: {e}")
    # If running in a restricted environment, you might need to use a pre-downloaded model or a different one.
    # For now, we'll stop if loading fails.
    raise

# Add a padding token if tokenizer doesn't have one
if tokenizer.pad_token is None:
    print("Setting pad_token to eos_token.")
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("Model loaded.")

# Prepare a sample prompt
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")
num_new_tokens_to_generate = 50

print(f"Input prompt: '{prompt}'")
print(f"Generating {num_new_tokens_to_generate} new tokens.")

Loading model and tokenizer for: gpt2...


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Setting pad_token to eos_token.
Model loaded.
Input prompt: 'The future of artificial intelligence is'
Generating 50 new tokens.


## CPU Profiling

**TODO:**
- Adapt the CPU profiling code from the demo.
- Ensure the inference is run within the `torch.profiler.profile` context.
- After the profiling block, add the code to print the top 5 CPU operators using `prof_cpu`.

In [4]:
print("\n--- Profiling on CPU ---")
cpu_device = torch.device("cpu")
model.to(cpu_device)
inputs_cpu = {k: v.to(cpu_device) for k, v in inputs.items()}

def run_cpu_inference(model_to_run, input_data, max_tokens, pad_id):
    # TODO: inference code to run the model
    return model_to_run.generate(
        input_ids=input_data["input_ids"],
        max_new_tokens=max_tokens,
        pad_token_id=pad_id,
        eos_token_id=-1
    )

print("Running inference on CPU and capturing profile...")
start_time_cpu_wall = time.time()

# TODO: Setup the torch.profiler.profile context manager for CPU
# Store the profiler object in 'prof_cpu'
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=False,
    profile_memory=False
) as prof_cpu: # Replace 'None' with the profiler setup
    # TODO: Add the record_function context manager (optional, but good practice)
    with torch.profiler.record_function("model_inference_cpu"):
        run_cpu_inference(model, inputs_cpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

end_time_cpu_wall = time.time()
cpu_wall_time = end_time_cpu_wall - start_time_cpu_wall
print(f"CPU Wall clock time: {cpu_wall_time:.4f} seconds")

# TODO: Add code here to print the key_averages table for prof_cpu,
# showing top 5 operators sorted by self_cpu_time_total

print("CPU Profiler Analysis (Top 5 Operators by Self CPU Time):")
print(prof_cpu.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))

`eos_token_id` should consist of positive integers, but is tensor([-1]). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.



--- Profiling on CPU ---
Running inference on CPU and capturing profile...


CPU Wall clock time: 5.9636 seconds
CPU Profiler Analysis (Top 5 Operators by Self CPU Time):
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          aten::addmm        66.32%        3.656s        66.72%        3.678s       1.533ms          2400  
                                             aten::mm        22.44%        1.237s        22.44%        1.237s      24.742ms            50  
                                  model_inference_cpu         5.36%     295.761ms       100.00%        5.513s        5.513s             1  
                                            aten::cat         1.22

## GPU Profiling

**TODO:**
- Adapt the GPU profiling code from the demo.
- Include the check for CUDA availability (`torch.cuda.is_available()`).
- Remember the warm-up run and `torch.cuda.synchronize()` for accurate timing.
- Ensure the inference is run within the `torch.profiler.profile` context, capturing both CPU and CUDA activities.
- After the profiling block, add the code to print the top 5 GPU operators using `prof_gpu`.

In [7]:
gpu_wall_time = -1.0 # Initialize in case GPU is not available

if torch.cuda.is_available():
    print("\n--- Profiling on GPU ---")
    gpu_device = torch.device("cuda")
    model.to(gpu_device)
    inputs_gpu = {k: v.to(gpu_device) for k, v in inputs.items()}
    print(f"CUDA device found: {torch.cuda.get_device_name(gpu_device)}")

    def run_gpu_inference(model_to_run, input_data, max_tokens, pad_id):
        with torch.no_grad():
            # TODO: add inference code here 
            # Remember to use torch.cuda.synchronize() before and after generation
            torch.cuda.synchronize()
            start_event.record()
            outputs = model_to_run.generate(
                input_data["input_ids"],
                attention_mask=input_data.get("attention_mask"),
                max_new_tokens=max_tokens,
                pad_token_id=pad_id
            )
            end_event.record()
            torch.cuda.synchronize()
            return None

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    print("Performing GPU warm-up run...")
    # TODO: Call run_gpu_inference for warm-up
    
    run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)
    print("Warm-up complete.")

    print("Running inference on GPU and capturing profile...")
    start_time_gpu_wall = time.time()
    

    # TODO: Setup the torch.profiler.profile context manager for GPU (CPU & CUDA activities)
    # Store the profiler object in 'prof_gpu'
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=False,
        profile_memory=False
    ) as prof_gpu:
        # TODO: Add the record_function context manager (optional)
        with torch.profiler.record_function("model_inference_gpu"):
            run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

    end_time_gpu_wall = time.time()
    gpu_wall_time = end_time_gpu_wall - start_time_gpu_wall
    print(f"GPU Wall clock time: {gpu_wall_time:.4f} seconds")

    gpu_e2e_latency_s_event = start_event.elapsed_time(end_event) / 1000.0
    gpu_tokens_per_second = num_new_tokens_to_generate / gpu_e2e_latency_s_event

    if torch.cuda.is_available():
        print(f"--- GPU Metrics ---")
        print(f"E2E Latency (CUDA Event): {gpu_e2e_latency_s_event:.4f} s")
        print(f"Tokens/sec: {gpu_tokens_per_second:.2f} tokens/s")

    print("GPU Profiler Analysis (Top 5 Operators by Self GPU Time):")
    print(prof_gpu.key_averages().table(sort_by="self_cuda_time_total", row_limit=5))

else:
    print("\nCUDA not available on this system. Skipping GPU profiling.")


--- Profiling on GPU ---
CUDA device found: Tesla T4
Performing GPU warm-up run...


Warm-up complete.
Running inference on GPU and capturing profile...
GPU Wall clock time: 1.1627 seconds
--- GPU Metrics ---
E2E Latency (CUDA Event): 0.6437 s
Tokens/sec: 77.68 tokens/s
GPU Profiler Analysis (Top 5 Operators by Self GPU Time):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                    model_inference_gpu         0.00%       0.000us         0.00%       0.000us       0.000us     642.667ms       359.24%     642

GPT-2 Medium

In [8]:
# TODO: Define the model name for GPT-2 (the smallest variant)
model_name = "gpt2-medium"  # Replace with "gpt2"

print(f"Loading model and tokenizer for: {model_name}...")
try:
    # TODO: Load the tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
except Exception as e:
    print(f"Error loading model {model_name}. Please ensure it's correct. Error: {e}")
    # If running in a restricted environment, you might need to use a pre-downloaded model or a different one.
    # For now, we'll stop if loading fails.
    raise

# Add a padding token if tokenizer doesn't have one
if tokenizer.pad_token is None:
    print("Setting pad_token to eos_token.")
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("Model loaded.")

# Prepare a sample prompt
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")
num_new_tokens_to_generate = 50

print(f"Input prompt: '{prompt}'")
print(f"Generating {num_new_tokens_to_generate} new tokens.")

Loading model and tokenizer for: gpt2-medium...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting pad_token to eos_token.
Model loaded.
Input prompt: 'The future of artificial intelligence is'
Generating 50 new tokens.


In [9]:
print("\n--- Profiling on CPU ---")
cpu_device = torch.device("cpu")
model.to(cpu_device)
inputs_cpu = {k: v.to(cpu_device) for k, v in inputs.items()}

def run_cpu_inference(model_to_run, input_data, max_tokens, pad_id):
    # TODO: inference code to run the model
    return model_to_run.generate(
        input_ids=input_data["input_ids"],
        max_new_tokens=max_tokens,
        pad_token_id=pad_id,
        eos_token_id=-1
    )

print("Running inference on CPU and capturing profile...")
start_time_cpu_wall = time.time()

# TODO: Setup the torch.profiler.profile context manager for CPU
# Store the profiler object in 'prof_cpu'
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=False,
    profile_memory=False
) as prof_cpu: # Replace 'None' with the profiler setup
    # TODO: Add the record_function context manager (optional, but good practice)
    with torch.profiler.record_function("model_inference_cpu"):
        run_cpu_inference(model, inputs_cpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

end_time_cpu_wall = time.time()
cpu_wall_time = end_time_cpu_wall - start_time_cpu_wall
print(f"CPU Wall clock time: {cpu_wall_time:.4f} seconds")

# TODO: Add code here to print the key_averages table for prof_cpu,
# showing top 5 operators sorted by self_cpu_time_total

print("CPU Profiler Analysis (Top 5 Operators by Self CPU Time):")
print(prof_cpu.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))

`eos_token_id` should consist of positive integers, but is tensor([-1]). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.



--- Profiling on CPU ---
Running inference on CPU and capturing profile...
CPU Wall clock time: 14.7562 seconds
CPU Profiler Analysis (Top 5 Operators by Self CPU Time):
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          aten::addmm        81.29%       11.274s        81.61%       11.319s       2.358ms          4800  
                                             aten::mm         9.87%        1.369s         9.87%        1.369s      27.375ms            50  
                                  model_inference_cpu         4.27%     591.981ms       100.00%       13.869s       13.869s      

In [10]:
gpu_wall_time = -1.0 # Initialize in case GPU is not available

if torch.cuda.is_available():
    print("\n--- Profiling on GPU ---")
    gpu_device = torch.device("cuda")
    model.to(gpu_device)
    inputs_gpu = {k: v.to(gpu_device) for k, v in inputs.items()}
    print(f"CUDA device found: {torch.cuda.get_device_name(gpu_device)}")

    def run_gpu_inference(model_to_run, input_data, max_tokens, pad_id):
        with torch.no_grad():
            # TODO: add inference code here 
            # Remember to use torch.cuda.synchronize() before and after generation
            torch.cuda.synchronize()
            start_event.record()
            outputs = model_to_run.generate(
                input_data["input_ids"],
                attention_mask=input_data.get("attention_mask"),
                max_new_tokens=max_tokens,
                pad_token_id=pad_id
            )
            end_event.record()
            torch.cuda.synchronize()
            return None

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    print("Performing GPU warm-up run...")
    # TODO: Call run_gpu_inference for warm-up
    
    run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)
    print("Warm-up complete.")

    print("Running inference on GPU and capturing profile...")
    start_time_gpu_wall = time.time()
    

    # TODO: Setup the torch.profiler.profile context manager for GPU (CPU & CUDA activities)
    # Store the profiler object in 'prof_gpu'
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=False,
        profile_memory=False
    ) as prof_gpu:
        # TODO: Add the record_function context manager (optional)
        with torch.profiler.record_function("model_inference_gpu"):
            run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

    end_time_gpu_wall = time.time()
    gpu_wall_time = end_time_gpu_wall - start_time_gpu_wall
    print(f"GPU Wall clock time: {gpu_wall_time:.4f} seconds")

    gpu_e2e_latency_s_event = start_event.elapsed_time(end_event) / 1000.0
    gpu_tokens_per_second = num_new_tokens_to_generate / gpu_e2e_latency_s_event

    if torch.cuda.is_available():
        print(f"--- GPU Metrics ---")
        print(f"E2E Latency (CUDA Event): {gpu_e2e_latency_s_event:.4f} s")
        print(f"Tokens/sec: {gpu_tokens_per_second:.2f} tokens/s")

    print("GPU Profiler Analysis (Top 5 Operators by Self GPU Time):")
    print(prof_gpu.key_averages().table(sort_by="self_cuda_time_total", row_limit=5))

else:
    print("\nCUDA not available on this system. Skipping GPU profiling.")


--- Profiling on GPU ---
CUDA device found: Tesla T4
Performing GPU warm-up run...
Warm-up complete.
Running inference on GPU and capturing profile...
GPU Wall clock time: 2.1285 seconds
--- GPU Metrics ---
E2E Latency (CUDA Event): 1.2148 s
Tokens/sec: 41.16 tokens/s
GPU Profiler Analysis (Top 5 Operators by Self GPU Time):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                    model_inference_gpu         0.00%       0

## 4. Deliverables & Analysis

**TODO:**

Based on the profiler tables you printed above, answer the following questions. Write your answers in this markdown cell.

**A. Top 5 CPU Operators:**
   1. Operator Name: `aten::addmm` | Self CPU Time %: `66.32%` | Likely Represents: Most of the linear work in the model, such as QKV calculations, attention heads output, and the MLP layers
   2. Operator Name: `aten::mm` | Self CPU Time %: `22.44%` | Likely Represents: Matrix multiplication without bias, probably mainly the final calculation that converts the last hidden state into vocabulary scores
   3. Operator Name: `model_inference_cpu` | Self CPU Time %: `5.36%` | Likely Represents: The custom label around the full inference process, so it contains all the smaller operations running inside of it
   4. Operator Name: `aten::cat` | Self CPU Time %: `1.22%` | Likely Represents: Joining tensors together while generating tokens, such as extending the output sequence and the growing ne3w entries for KV vectors for new tokens
   5. Operator Name: `aten::index_select` | Self CPU Time %: `0.51%` | Likely Represents: Selecting certain values from tensors using their indices during the generation process


**B. Top 5 GPU Operators (if CUDA was available):**
   1. Operator Name: `model_inference_gpu` | Self CUDA Time %: `359.24%` | Likely Represents: The label around the whole GPU inference, and the percentage is above 100% because it overlaps with the operations running inside it, not because GPU usage was actually 359%
   2. Operator Name: `aten::addmm` | Self CUDA Time %: `46.86%` | Likely Represents: The linear calculations with bias, mainly QKV, attention heads output, and the two MLP projections
   3. Operator Name: `internal::gemvx` | Self CUDA Time %: `30.92%` | Likely Represents: A low-level CUDA kernel doing some of the matrix-vector calculations used by the linear layers
   4. Operator Name: `aten::mm` | Self CUDA Time %: `17.50%` | Likely Represents: Matrix multiplication without bias, probably mainly calculating the vocabulary scores for every newly generateds token
   5. Operator Name: `gemv2T_kernel` | Self CUDA Time %: `16.77%` | Likely Represents: low-level CUDA kernel used internally for matrix-vector calculations


**C. Wall Clock Time Comparison (gpt2 vs gpt2-medium from demo):**
   - CPU Wall Time (`gpt2` from this exercise): `5.9636` seconds
   - CPU Wall Time (`gpt2-medium` from demo): `14.7562` seconds
   - GPU Wall Time (`gpt2` from this exercise): `1.1627` seconds
   - GPU Wall Time (`gpt2-medium` from demo): `2.1285` seconds

- Did `gpt2` run faster than `gpt2-medium` on both CPU and GPU as expected? `GPT-2 was faster on both CPU and GPU, which makes sense because GPT-2 Medium is a bigger model that has more layers, a bigger hidden size which means bigger mm, and more parameters to process for every generated token`

**D. Brief Interpretation of Top Operators:**
   - What kind of operations generally dominate the top spots on both CPU and GPU for this Transformer model?
   `Matrix multiplication operations mostly dominate both, especially addmm and mm, since most of the models work comes from the QKV projections, attention output, MLP layers, and final vocabulary projection`
   - Were there any surprising operators in the top 5 for either CPU or GPU?
   `Yes, aten::addmm and aten::mm ratio to other operations got really minimized on GPU (compared to ratios of mm to other operation on CPU), and this is expecteds as the GPU does mm much faster than GPU`